# Model Output Collection Examples (including FlowGraph)

This notebook demonstrates `pws.Output` with a `FlowGraph` model that includes
a STARFIT reservoir node. We'll reproduce the final plot of notebook `06_flow_graph_starfit.ipynb` but we'll only output 
data on locations of interest. 

We'll demonstrate the capabilities of the `Output` object to 

1. Output monthly accumulations on ALL spatial locations
2. Output full-timeseries of variables on "HRUs of interest" ("hoi") and "nodes of interest" ("noi"), including the additional FlowGraph output variables
3. Calculate arbitrary, user-defined and user-supplied statistics on these full timeseries on locations of interest. 
4. Organize the results in a clear way, suppliying xarray DataArrays with dimensions, coordinates, and metadata.

We'll show two different ways to initialize and manage the Output object. The first way will be to explicity manage it, the second way will simply pass the relevant arguments or configuration through the `Model` object and access it through the `Model` instance.  

## Setup
We'll define the packages, paths, etc that we need. 

In [ ]:
import pathlib as pl
from pprint import pprint

import jupyter_black
import numpy as np
import pywatershed as pws
import xarray as xr

jupyter_black.load()  # auto-format the code in this notebook

In [ ]:
nb_output_dir = pl.Path("./model_output_flowgraph_example")

pkg_root = pws.constants.__pywatershed_root__
big_sandy_param_file = pkg_root / "data/big_sandy_starfit_parameters.nc"
sf_params = pws.Parameters.from_netcdf(big_sandy_param_file, use_xr=True)
sfp_ds = sf_params.to_xr_ds().copy()

# We'll increase the reservoir capacity by 50%
cap_mult = 1.5
sfp_ds["GRanD_CAP_MCM"] *= cap_mult
sf_params_new = pws.Parameters.from_ds(sfp_ds)

domain_dir = pkg_root / "data/pywatershed_addtl_domains/fgr_2yr"
control_file = domain_dir / "nhm.control"

# Number of days to run
ndays_run = 365 * 2

params_file_channel = domain_dir / "parameters_PRMSChannel.nc"
params_channel = pws.parameters.PrmsParameters.from_netcdf(params_file_channel)

dis_file = domain_dir / "parameters_dis_hru.nc"
dis_hru = pws.Parameters.from_netcdf(dis_file, encoding=False)

dis_both_file = domain_dir / "parameters_dis_both.nc"
dis_both = pws.Parameters.from_netcdf(dis_both_file, encoding=False)

# Build multi-process model dictionary (without channel)
nhm_processes = [
    pws.PRMSSolarGeometry,
    pws.PRMSAtmosphere,
    pws.PRMSCanopy,
    pws.PRMSSnow,
    pws.PRMSRunoff,
    pws.PRMSSoilzone,
    pws.PRMSGroundwater,
]

Since we'll run the model twice, we'll create some helper function to provide fresh inputs.

In [ ]:
def get_control():
    control = pws.Control.load_prms(control_file, warn_unused_options=False)
    if ndays_run > 0:
        control.edit_n_time_steps(ndays_run)

    control.options = control.options | {
        "input_dir": domain_dir,
        "budget_type": "warn",
        "calc_method": "numba",
    }

    if "netcdf_output_dir" in control.options:
        del control.options["netcdf_output_dir"]
    if "netcdf_output_var_names" in control.options:
        del control.options["netcdf_output_var_names"]

    return control


def get_model_dict(control: pws.Control):
    model_dict = {
        "control": control,
        "dis_both": dis_hru,
        "dis_hru": dis_both,
        "model_order": [],
    }

    for proc in nhm_processes:
        proc_name = proc.__name__
        proc_rename = "prms_" + proc_name[4:].lower()
        model_dict["model_order"] += [proc_rename]
        model_dict[proc_rename] = {}
        proc_dict = model_dict[proc_rename]
        proc_dict["class"] = proc
        proc_param_file = domain_dir / f"parameters_{proc_name}.nc"
        proc_dict["parameters"] = pws.Parameters.from_netcdf(proc_param_file)
        proc_dict["dis"] = "dis_hru"

    model_dict = pws.prms_channel_flow_graph_to_model_dict(
        model_dict=model_dict,
        prms_channel_dis=dis_both,
        prms_channel_dis_name="dis_both",
        prms_channel_params=params_channel,
        new_nodes_maker_dict={
            "starfit": pws.hydrology.starfit.StarfitFlowNodeMaker(
                None,
                sf_params_new,
                budget_type="warn",
                compute_daily=False,
            )
        },
        new_nodes_maker_names=["starfit"],
        new_nodes_maker_indices=[0],
        new_nodes_maker_ids=[999],
        new_nodes_flow_to_nhm_seg=[44426],
        graph_budget_type="warn",
        addtl_output_vars=["spill", "release"],
        prms_channel_node_maker_name="prms_channel",  # this is the default
    )
    return model_dict

## Custom time statistics
Here we can define our own stats that we want to calculate on the data. We can also import statistics from pws.

In [ ]:
from pywatershed.analysis.time_stats import (
    median_monthly,
    seven_day_mean_calendar_year_max,
)


# Define custom statistics functions
def mean_monthly(da: xr.DataArray) -> xr.DataArray:
    return da.resample(time="1MS").mean(dim="time")


def rolling_7day_mean_wy_max(da: xr.DataArray):
    """
    This function is also available like seven_day_mean_calendar_year_max
    but this shows some of the guts of how to build such a function.
    """

    time_window: int = 7
    seasons: str = ["ONDJFMAMJJAS"]

    roll_mean = da.rolling(
        time=time_window, min_periods=None, center=True
    ).mean()

    wy_max = roll_mean.resample(time=xr.groupers.SeasonResampler(seasons)).max(
        dim="time", skipna=True
    )
    wy_max_dates = roll_mean.resample(
        time=xr.groupers.SeasonResampler(seasons)
    ).map(lambda x: x.idxmax(dim="time", skipna=True))

    wy_max["time"] = wy_max_dates

    return wy_max

## Model with external Output object

We must pass an instantiated `Model` to `Output`, so we must do this first. 

In [ ]:
control = get_control()
model = pws.Model(get_model_dict(control))

Now we will define the Output object.  For HOIs there are only `nhm_id`s being used to identify locations. 
For NOIs, if using `PRMSChannel`, `nhm_seg` is used to identify locations. When a `FlowGraph` is being used, the location identification is 2-dimensional: we must identify both the `node_maker_name`s and the `node_maker_id`s for the "nodes of interest" (noi). 

In [ ]:
hoi = ([dis_hru.parameters["nhm_id"][0].tolist()],)  # 85980

noi_res_and_below_ids = [
    ("prms_channel", 44426),  # this is the node below the reservoir
    ("starfit", 999),  # this is the Big Sandy reservoir node
]

noi_res_ids = [
    ("starfit", 999),
]

output = pws.base.Output(
    control=control,
    model=model,
    # monthly accumulation
    monthly_accum_var_list=["node_outflows", "pkwater_equiv"],
    # hoi
    hoi_var_list=["hru_actet", "pkwater_equiv"],
    hoi_ids=hoi,
    hoi_stats={
        mean_monthly: ["hru_actet", "pkwater_equiv"],
        rolling_7day_mean_wy_max: ["pkwater_equiv"],
        seven_day_mean_calendar_year_max: ["pkwater_equiv"],
    },
    # noi
    noi_ids={
        "node_outflows": noi_res_and_below_ids,
        "node_storages": noi_res_ids,
        "spill": noi_res_ids,
        "release": noi_res_ids,
    },
    noi_stats={
        mean_monthly: ["node_outflows", "release"],
        median_monthly: ["node_outflows"],
        rolling_7day_mean_wy_max: [
            "node_outflows",
            "node_storages",
            "spill",
            "release",
        ],
    },
)

In instantiating the `Output` object, after the `control` and `model` args, three separate sections are highlighted: monthly accumulations, hoi, and noi. The monthly acumulation section is straight forward. You can only really request variables to be acumulated on a monthly basis and, as we'll see later, this trigers the solution of a `n_days_per_month` attribute out `output`. As for the hoi and the noi sections, notice first that different styles of requests are being made. For hoi, we are specifying 3 arguments with separate `hoi_var_list` and `hoi_ids` lists. In this separate list case, it is assumed that all variables use the samd hoi_ids. In the noi section, the `noi_ids` argument is a dict allowing specification of `variable: noi`, so not the same noi for all the variables. Either style of request can be used in both the hoi and noi sections, but they can not both be used in a given section. After the variables and ids are handled, the stats are requested via dictionary where stat functions are passed as keys and the values are list variables on which to compute the stat. We will see the results momentarily.

Now we will pass the `output_obj` to the `model.run` method. 

In [ ]:
model.run(finalize=True, output_obj=output)

### Results
First, let's look at the monthly_accumulations in the output object.

In [ ]:
print(list(output.monthly_accumulations.keys()))

In [ ]:
display(output.monthly_accumulations["node_outflows"])

As mentioned, we also track the number of days accumulated in each month. Partial months will be reflected, so that the
mean of the month is always obtained by dividing the accumulated variable by the `n_days_per_month` atrribute. The month 
coordinate for each month is time stampped with the first day of the month. 

In [ ]:
display(output.n_days_per_month)

For HOI and NOI requests, it is important to understand that the full-timeseries data are kept in the object for the locations of interest. These full-timeseries can be accessed as so:

In [ ]:
for aa in output.hoi_arrays:
    display(output.hoi_arrays[aa])

In [ ]:
for aa in output.noi_arrays:
    display(output.noi_arrays[aa])

Notice that for `node_outflows` two locations were requested and returned in the full-timeseries. It is important to note that the full-timeseries data are available. Many or most statistcs require having the full timeseries so keeping all these values is unavoidable. But this also permits to calculate new statistics after the fact, beyond the requested stats when `Output` is instantiated. Note that all the stats requested are essentially done as a post process on the data collection while the model is running. 

Now let's look at the stats that were requested up-front.

In [ ]:
for var in output.hoi_stats:
    for stat in output.hoi_stats[var]:
        print(f"{var}: {stat}")
        display(output.hoi_stats[var][stat])

Notice that each returned statistic is a named `xr.DataArray`, named with the variable name. The metadata, or attr, shows the statistic computed as well as the period of record. 

Notice the two stats for `pkwater_equiv`, they are calculated on two different bases: calendar year and water year. The former has two results because there were two full years whereas the later only has one result because there is only a single full water year in the period. 

The structure for the NOI stats is similarly hierarchical, organized by variable and then statistic name. 

In [ ]:
for var in output.noi_stats:
    for stat in output.noi_stats[var]:
        print(f"{var}: {stat}")
        display(output.noi_stats[var][stat])

As requested, `node_outflows` timeseries are returned on 2 locations or nodes whereas the other variables are returned on a single location (the Big Sandy reservoir).

## Model with internal Output object

Now we let the `Model` object handle the `Output` object for us. We still need to pass all the same information as before, except for we do not need to pass the Control or Model objects. The information is passed to `Model` at initialization via the argument `output_obj_kwargs_dict`. 

In [ ]:
output_obj_kwargs_dict = {
    "monthly_accum_var_list": ["hru_actet", "pkwater_equiv"],
    # hois
    "hoi_var_list": ["hru_actet", "pkwater_equiv"],
    "hoi_ids": hoi,
    "hoi_stats": {
        mean_monthly: ["hru_actet", "pkwater_equiv"],
        rolling_7day_mean_wy_max: ["pkwater_equiv"],
        seven_day_mean_calendar_year_max: ["pkwater_equiv"],
    },
    # nois
    "noi_ids": {
        "node_outflows": noi_res_and_below_ids,
        "node_storages": noi_res_ids,
        "spill": noi_res_ids,
        "release": noi_res_ids,
    },
    "noi_stats": {
        mean_monthly: ["node_outflows", "release"],
        median_monthly: ["node_outflows"],
        rolling_7day_mean_wy_max: [
            "node_outflows",
            "node_storages",
            "spill",
            "release",
        ],
    },
}
control2 = get_control()
model2 = pws.Model(
    get_model_dict(control2), output_obj_kwargs_dict=output_obj_kwargs_dict
)

In [ ]:
model2.run(finalize=True)

### Results
We expect the output to be exactly the same as when we ran the model above. We just have to access the output in a slightly different place, as the `output_obj` attribute on the `Model` object. 

In [ ]:
model2.output_obj.monthly_accumulations.keys()

In [ ]:
model2.output_obj.hoi_stats.keys()

In [ ]:
model2.output_obj.hoi_stats["pkwater_equiv"].keys()

In [ ]:
model2.output_obj.noi_stats.keys()

In [ ]:
model2.output_obj.noi_stats["spill"].keys()

In [ ]:
output2 = model2.output_obj
xr.testing.assert_equal(
    output.noi_stats["spill"]["rolling_7day_mean_wy_max"],
    output2.noi_stats["spill"]["rolling_7day_mean_wy_max"],
)

In [ ]:
the_mean_hru_actet = (
    output2.monthly_accumulations["hru_actet"].set_xindex("nhm_id")
).sel(
    nhm_id=output2.hoi_stats["hru_actet"]["mean_monthly"].nhm_id
) / output.n_days_per_month

In [ ]:
np.testing.assert_allclose(
    the_mean_hru_actet.values,
    output2.hoi_stats["hru_actet"]["mean_monthly"].values,
)

## The final plot
... from notebook 06_flow_graph_starfit.ipynb can be created while writing much less output from the model. Of course, we expect it to match the plot in the other notebook, which it does. 

In [ ]:
plot_data = xr.merge(
    [
        output2.noi_arrays["node_outflows"]
        .set_xindex("node_maker_id")
        .sel(node_maker_id=999),
        output2.noi_arrays["spill"],
        output2.noi_arrays["release"],
    ],
    compat="override",
).rename(
    {
        "node_outflows": f"Big Sandy CAP*{cap_mult}",
        "spill": f"Big Sandy Spill CAP*{cap_mult}",
        "release": f"Big Sandy Release CAP*{cap_mult}",
    }
)

import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # Temporarily suppress all warnings
    import pandas as pd
    import hvplot.xarray  # noqa, after xr
    import hvplot.pandas  # noaq, after pandas

    display(
        plot_data.drop_vars(
            ["node_maker_name", "node_maker_id", "node_coord"]
        ).hvplot(
            width=1200,
            height=500,
            ylabel="streamflow (cfs)",
        )
    )